# mlbt — Train short-horizon prediction models on Google Colab

End-to-end run on a Colab T4/A100/L4:
1. Clone the repo, install deps
2. Collect 2y of multi-source data (free APIs, no signups)
3. Build the aligned feature matrix + targets
4. Train **LightGBM** baseline, **LSTM**, and **PatchTST-Lite** transformer
5. Walk-forward evaluate, run the backtester, print summary stats

Switch to GPU runtime: **Runtime → Change runtime type → T4 GPU (or better)**.
If you only need the GBM baseline, CPU is fine.

In [ ]:
# --- Clone repo (replace with your fork if needed) ---
import os, subprocess, sys
if not os.path.exists('/content/ML_Based_Trading'):
    !git clone --depth 1 https://github.com/euryalus-kay/ml_based_trading.git /content/ML_Based_Trading
%cd /content/ML_Based_Trading

In [ ]:
# --- Install dependencies ---
!pip install -q -r requirements.txt
!pip install -q lightgbm torch torchvision torchaudio
!pip install -q -e .

In [ ]:
# --- Verify install + list sources ---
!python -m mlbt.cli list-sources

In [ ]:
# --- Pick collection window ---
# Intraday yfinance only goes back ~30d for 1m and ~60d for 5m.
# For longer history, use daily bars (decades available).
import datetime as dt
END = dt.date.today().isoformat()
START = (dt.date.today() - dt.timedelta(days=55)).isoformat()  # ~55d window for 5m bars
print('Window:', START, '->', END)

In [ ]:
# --- Collect raw data ---
!python -m mlbt.cli collect --start {START} --end {END}

In [ ]:
# --- Build dataset (features + targets) ---
!python -m mlbt.cli build-dataset --start {START} --end {END} --bar 5min \
  --out data/dataset.parquet --horizons 1,3,6,12

In [ ]:
# --- Quick look at the dataset ---
import pandas as pd
df = pd.read_parquet('data/dataset.parquet')
print(df.shape)
print('symbols:', df['symbol'].nunique() if 'symbol' in df else None)
print('span:', df.index.min(), '->', df.index.max())
df.tail()

In [ ]:
# --- Train GBM baseline (CPU, fast) ---
!python -m mlbt.cli train --dataset data/dataset.parquet --target y_up_3 --model gbm --out data/model_gbm

In [ ]:
# --- Backtest the GBM model ---
!python -m mlbt.cli backtest --dataset data/dataset.parquet --model-dir data/model_gbm \
  --out data/backtest_gbm.html
import json
print(json.dumps(json.loads(open('data/backtest_gbm.json').read()), indent=2))

In [ ]:
# --- Train LSTM (GPU recommended) ---
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from mlbt.ml.train import train_model
metrics = train_model(
    dataset_path='data/dataset.parquet',
    target='y_up_3',
    model='lstm',
    out_dir='data/model_lstm',
    window=64, epochs=10,
)
metrics

In [ ]:
# --- Train PatchTST-Lite transformer (best signal usually) ---
metrics_pt = train_model(
    dataset_path='data/dataset.parquet',
    target='y_up_3',
    model='patchtst',
    out_dir='data/model_patchtst',
    window=64, epochs=12,
)
metrics_pt

In [ ]:
# --- Plot equity curve from the GBM backtest ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt
preds = pd.read_parquet('data/model_gbm/predictions.parquet')
ds = pd.read_parquet('data/dataset.parquet')[['y_logret_3']]
j = preds.join(ds, how='inner').dropna()
pos = (2*(j['y_score']-0.5)).clip(-1, 1)
pos = pos.where((j['y_score']-0.5).abs() > 0.02, 0.0)
ret = pos * j['y_logret_3'] - pos.diff().abs().fillna(pos.abs()) * 2e-4
(1 + ret).cumprod().plot(figsize=(10,4), title='Net equity (incl. 2 bps cost)')
plt.grid(True); plt.show()
print('Directional accuracy on traded bars:', (np.sign(pos[pos!=0])==np.sign(j.loc[pos!=0,'y_logret_3'])).mean())